In [ ]:
!pip install python-telegram-bot

In [ ]:
from google.colab import files

# Загрузи math_misconception_model.pkl и answers.pkl (оба из model1)
uploaded = files.upload()

In [ ]:
import joblib

# грузим и модель, и словарь ответов из .pkl (как сохраняли в model1)
model = joblib.load("math_misconception_model.pkl")
answers = joblib.load("answers.pkl")

print("✅ Model loaded!")
print("✅ Answers loaded:", len(answers), "классов")

In [ ]:
from telegram import Update
from telegram.ext import Application, CommandHandler, MessageHandler, filters, ContextTypes
from getpass import getpass

# ✅ ФИКС 1: токен не в коде — вводим руками, в файле не остаётся
TOKEN = getpass("Вставь новый токен бота (@BotFather): ")


async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        "🤖 Привет! Пришли объяснение ученика — я определю тип ошибки.\n\n"
        "Если хочешь точнее: отправь две строки —\n"
        "1) текст вопроса\n"
        "2) объяснение ученика"
    )


def build_text(message_text: str) -> str:
    """✅ ФИКС 3: собираем вход так же, как в обучении:
    QuestionText + ' ' + StudentExplanation.
    Если пользователь прислал две строки — считаем их вопросом и объяснением,
    иначе используем весь текст как объяснение."""
    lines = [l.strip() for l in message_text.split("\n") if l.strip()]
    if len(lines) >= 2:
        question, explanation = lines[0], " ".join(lines[1:])
        return f"{question} {explanation}"
    return message_text


async def message(update: Update, context: ContextTypes.DEFAULT_TYPE):
    text = update.message.text

    # ✅ базовая проверка ввода
    if not text or not text.strip():
        await update.message.reply_text("Пришли текст объяснения 🙂")
        return

    model_input = build_text(text)
    prediction = model.predict([model_input])[0]

    # ✅ ФИКС 2: результат в ОТДЕЛЬНУЮ переменную — не затираем словарь answers
    explanation = answers.get(
        prediction,
        "Пока нет объяснения для этого класса."
    )

    await update.message.reply_text(
        f"🧠 Prediction: {prediction}\n\n"
        f"💬 Explanation: {explanation}"
    )


app = Application.builder().token(TOKEN).build()
app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, message))

import nest_asyncio
nest_asyncio.apply()

await app.initialize()
await app.start()
await app.updater.start_polling()

print("🤖 Бот запущен! Теперь напиши ему сообщение в Telegram и проверь ответ.")